### RAG Pipelines - Data Ingestion to Vector DB pipeline

In [32]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [33]:
from langchain_core import documents
## data ingestion - reading all the pdfs

def process_all_pdfs(pdf_directory):
    """Process all the PDF files in directory"""

    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all Pdf files recursively

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} pdf files to process")

    for pdf_file in pdf_files:
        print(f"\nProcesssing : {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source info to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"✓ Loaded {len(documents)} pages")
        
        except Exception as e:
            print(f"✗ Error:{e}")

    print(f"\n Total documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")


Found 3 pdf files to process

Processsing : ragg.pdf
✓ Loaded 7 pages

Processsing : rag_arch.pdf
✓ Loaded 8 pages

Processsing : rag_article.pdf
✓ Loaded 19 pages

 Total documents loaded: 34


In [34]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m120', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) obsidian/1.5.12 Chrome/120.0.6099.283 Electron/28.2.3 Safari/537.36', 'creationdate': '2024-09-29T07:50:14+00:00', 'moddate': '2024-09-29T07:50:14+00:00', 'source': '..\\data\\pdf\\ragg.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'ragg.pdf', 'file_type': 'pdf'}, page_content='Retrieval Augmented Generation\n"RAG takes input and retrieves a set of relevant/supporting documents given a source (e.g.,\nWikipedia). The documents are concatenated as context with the original input prompt and fed\nto the text generator which produces the final output. This makes RAG adaptive for situations\nwhere facts could evolve over time. This is very useful as Large Language Models\'s parametric\nknowledge is static. RAG allows language models to bypass retraining, enabling access to the\nlatest information for generating reliable outputs v

In [35]:
from sqlalchemy import text
from langchain_core.messages.utils import _default_text_splitter
## Text splitting - into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ","",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content {split_docs[0].page_content[:200]}")
        print(f"Metadata {split_docs[0].metadata}")

    return split_docs

In [36]:
chunks = split_documents(all_pdf_documents)
chunks

Split 34 documents into 59 chunks

Example chunk:
Content Retrieval Augmented Generation
"RAG takes input and retrieves a set of relevant/supporting documents given a source (e.g.,
Wikipedia). The documents are concatenated as context with the original input
Metadata {'producer': 'Skia/PDF m120', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) obsidian/1.5.12 Chrome/120.0.6099.283 Electron/28.2.3 Safari/537.36', 'creationdate': '2024-09-29T07:50:14+00:00', 'moddate': '2024-09-29T07:50:14+00:00', 'source': '..\\data\\pdf\\ragg.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'ragg.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m120', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) obsidian/1.5.12 Chrome/120.0.6099.283 Electron/28.2.3 Safari/537.36', 'creationdate': '2024-09-29T07:50:14+00:00', 'moddate': '2024-09-29T07:50:14+00:00', 'source': '..\\data\\pdf\\ragg.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'ragg.pdf', 'file_type': 'pdf'}, page_content='Retrieval Augmented Generation\n"RAG takes input and retrieves a set of relevant/supporting documents given a source (e.g.,\nWikipedia). The documents are concatenated as context with the original input prompt and fed\nto the text generator which produces the final output. This makes RAG adaptive for situations\nwhere facts could evolve over time. This is very useful as Large Language Models\'s parametric\nknowledge is static. RAG allows language models to bypass retraining, enabling access to the\nlatest information for generating reliable outputs v

## Embedding and VectorStore

In [37]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [38]:
class EmbeddingManager:
    """Handles document embedding generation using SenctenceTrasformer"""

    def __init__(self,model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name : HugginFace model name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None # we will set this later
        self._load_model() #_ underscore because this is a protected class and can be used under this function only

    def _load_model(self):
        """Load the SenctenceTransformer model"""

        try:
            print(f"Loading embedding model{self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension:{self.model.get_sentence_embedding_dimension()}")

        except Exception as e:
            print(f"Error loading model {self.model_name}:{e}")
            raise

    def generate_embeddings(self,texts:List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args: 
            texts: List of text strings to embed

        Returns:
        numpy array of embeddings with shape (len(texts), embedding_dim)

        """

        if not self.model:
            raise ValueError("model not loaded")

        print(f"Generatig embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

    # just to get the embedding dimension 
    # def get_embedding_dimension(self):
    #     """Get the embedding dimension of the model"""
    #     if not self.model:
    #         raise ValueError("Model not loaded")
    #         return self.model.get_sentence_embedding_dimension()


## we can initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding modelall-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2784.99it/s]


Model loaded successfully. Embedding dimension:384


C:\Users\yashb\AppData\Local\Temp\ipykernel_16404\2821436103.py:22: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension:{self.model.get_sentence_embedding_dimension()}")


### 

### VectorStore

In [39]:
from langchain_community.document_loaders.parsers.images import numpy
from typing import List,Any
import numpy as np

class VectorStore:
    """Manages document embeddings in a chromaDB vector store"""
    
    def __init__(self,collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """ initialize vector store
        Args :
            collection_name : Name of teh chromaDB collection 
            persist directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Intialize chromadb client and collection"""
                            
        try:
            #Create persistent chromadb client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized,. Collection : {self.collection_name}")
            print(f"Existing documents in collection : {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store:{e}")
            raise

    def add_documents(self, documents: List[Any],embeddings:np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args: 
            documents: List of Langchain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents)!= len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc,embedding) in enumerate(zip(documents,embeddings)):
            
            #Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            #Document content
            documents_text.append(doc.page_content)

            #Embedding
            embeddings_list.append(embedding.tolist())


            try:
                self.collection.add(
                    ids = ids,
                    embeddings = embeddings_list,
                    metadatas = metadatas,
                    documents = documents_text
                )
                print(f"Successfully added {len(documents)} documents to vector store")
                print(f"Total documents in collection: {self.collection.count()}")
            
            except Exception as e:
                print(f"Error adding documents to vector store: {e}")
                raise

vectorstore = VectorStore()
vectorstore




Vector store initialized,. Collection : pdf_documents
Existing documents in collection : 59


In [40]:
chunks

[Document(metadata={'producer': 'Skia/PDF m120', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) obsidian/1.5.12 Chrome/120.0.6099.283 Electron/28.2.3 Safari/537.36', 'creationdate': '2024-09-29T07:50:14+00:00', 'moddate': '2024-09-29T07:50:14+00:00', 'source': '..\\data\\pdf\\ragg.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'ragg.pdf', 'file_type': 'pdf'}, page_content='Retrieval Augmented Generation\n"RAG takes input and retrieves a set of relevant/supporting documents given a source (e.g.,\nWikipedia). The documents are concatenated as context with the original input prompt and fed\nto the text generator which produces the final output. This makes RAG adaptive for situations\nwhere facts could evolve over time. This is very useful as Large Language Models\'s parametric\nknowledge is static. RAG allows language models to bypass retraining, enabling access to the\nlatest information for generating reliable outputs v

In [ ]:
### Convert chunks into embeddings
texts = [doc.page_content for doc in chunks]

# Generate the Embeddings

embeddings = embedding_manager.generate_embeddings(texts)

# Store it in vectorDB

vectorstore.add_documents(chunks,embeddings)

Generatig embeddings for 59 texts...


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.27it/s]

### Rag Retrieval Pipeline from vectorstore 

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [ ]:
rag_retriever

In [ ]:
rag_retriever.retrieve("An LLM agent controls the retrieval process entirely")

Retrieving documents for query: 'An LLM agent controls the retrieval process entirely'
Top K: 5, Score threshold: 0.0
Generatig embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 68.90it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_b794777f_38',
  'content': 'questions the model already knows skip retrieval entirely. Medium-complexity\nquestions get one pass. Genuinely complex multi-hop questions trigger iterative\nretrieval rounds. At the scale of millions of daily queries, reducing unnecessary\nretrieval saves significant compute and also reduces retrieval noise — fewer\nirrelevant passages contaminating responses to questions the model already had the\nanswer to. (arXiv:2403.14403)\n8. Agentic RAG — retrieval as a reasoning loop\nAll previous RAG types follow a predetermined script: retrieve once (or a fixed\nnumber of times), then generate. Agentic RAG abandons the script. An LLM agent\ncontrols the retrieval process entirely: decide what it needs to know, choose which\ntool to invoke (vector search, web search, SQL query, API call), evaluate the result,\ndecide whether to continue searching, and iterate until it is satisfied. A complex\nquestion might trigger five iterations across three differe

In [ ]:
rag_retriever.retrieve("Multimodal RAG — beyond text")

Retrieving documents for query: 'Multimodal RAG — beyond text'
Top K: 5, Score threshold: 0.0
Generatig embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 83.32it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_2c61167a_40',
  'content': 'the model’s own reasoning trajectory rather than orchestrated externally. Best for:\ndeep research tasks, competitive intelligence, autonomous document workflows.\nMultimodal RAG — beyond text\nMultimodal RAG extends the index beyond text to images, tables, charts, diagrams,\nand audio. Vision models process visual content, OCR handles structured tables,\nand the retrieval pipeline can return image patches or data extracts alongside text.\nFor domains where knowledge lives in formats other than prose, this is not optional\n— it is essential. Best for: medical imaging, engineering documentation, financial\nreports heavy with charts and tables.\nHow to pick the right type\n4/12/26, 3:34 PM Retrieval-Augmented Generation (RAG): The Complete Guide | by Kunal Khanvilkar | Apr, 2026 | Medium\nhttps://medium.com/p/83e700b6f1ac?postPublishedType=initial 7/19',
  'metadata': {'doc_index': 40,
   'total_pages': 19,
   'creationdate': '2026-04-12T19:34:11+

### Integration Vectordb context pipeline with LLM output

In [ ]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="gemma2-9b-it",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content